## Setup

In [6]:
import sys
from pathlib import Path
import duckdb

In [7]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))    ## puts the project root on Python’s import path

RAW = ROOT/"data"/"raw"
con = duckdb.connect()

def src(name):
    """Return the DuckDB read expression for data/raw/<name>.csv."""
    path = (RAW / f"{name}.csv").as_posix()
    return f"read_csv_auto('{path}')"

def q(title, sql):
    """Run a query and print its result under a title."""
    print(f"\n--- {title}")
    con.sql(sql).show(max_rows=100)

def empty_counts(name):
    """Print, per column, how many values are NULL or blank."""
    cols = [r[0] for r in con.sql(f"DESCRIBE SELECT * FROM {src(name)}").fetchall()]
    parts = ", ".join(
        f"COUNT(*) FILTER (WHERE {c} IS NULL OR TRIM(CAST({c} AS VARCHAR)) = '') AS {c}"
        for c in cols
    )
    q(f"{name}: empty values per column", f"SELECT {parts} FROM {src(name)}")

## Raw Profile

In [8]:
print("Raw folder:", RAW)
print("CSV files:", len(list(RAW.glob("*.csv"))))

for f in sorted(RAW.glob("*.csv")):
    source = f"read_csv_auto('{f.as_posix()}')"
    n = con.sql(f"SELECT COUNT(*) FROM {source}").fetchall()[0][0]
    print(f"\n{f.name}: {n:,} rows")
    con.sql(f"DESCRIBE SELECT * FROM {source}").show()

Raw folder: D:\DEProjects\retail-lakehouse\data\raw
CSV files: 8

campaign_desc.csv: 30 rows
┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ DESCRIPTION │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ CAMPAIGN    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ START_DAY   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ END_DAY     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘


campaign_table.csv: 7,208 rows
┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼───────────

## Transactions exploration

In [9]:
T = src("transaction_data")

q("nulls per column", f"""
SELECT
COUNT(*) FILTER (WHERE household_key IS NULL) AS hh_null,
COUNT(*) FILTER (WHERE BASKET_ID IS NULL) AS basket_null,
COUNT(*) FILTER (WHERE PRODUCT_ID IS NULL) AS prod_null,
COUNT(*) FILTER (WHERE SALES_VALUE IS NULL) AS sales_null,
FROM {T}
""")

q("duplicate rows", f"""
SELECT COUNT(*) AS total_rows,
(SELECT COUNT(*) FROM (SELECT DISTINCT * FROM {T})) AS distinct_rows
FROM {T}
""")

q("is (BASKET_ID, PRODUCT_ID) unique?", f"""
SELECT COUNT(*) AS total_rows,
COUNT(DISTINCT (BASKET_ID, PRODUCT_ID)) AS distinct_keys
FROM {T}
""")

q("DAY Range", f"""
SELECT MIN(DAY), MAX(DAY), COUNT(DISTINCT DAY) AS distinct_days
FROM {T}
""")

q("suspicious values", f"""
SELECT
COUNT(*) FILTER (WHERE QUANTITY < 0) AS negative_quantity,  -- Means returns or reversals
COUNT(*) FILTER (WHERE QUANTITY = 0) AS zero_quantity,      -- Scans with no units
COUNT(*) FILTER (WHERE SALES_VALUE < 0) AS negative_sales,  -- refunds or reversed revenue
COUNT(*) FILTER (WHERE SALES_VALUE = 0) AS zero_sales,      -- no money received, free or 100% discount
FROM {T}
""")

q("TRANS_TIME samples", f"""
SELECT DISTINCT TRANS_TIME
FROM {T} LIMIT 8
""")

q("zero sales but items sold", f"""
SELECT COUNT(*)
FROM {T} WHERE SALES_VALUE = 0 AND QUANTITY > 0
""")

q("zero-qty vs zero-sales", f"""
SELECT
(QUANTITY = 0) AS zero_qty,
(SALES_VALUE = 0) AS zero_sales,
COUNT(*) AS n_rows
FROM {T}
WHERE QUANTITY = 0 OR SALES_VALUE = 0
GROUP BY 1, 2
ORDER BY 1, 2
""")


--- nulls per column
┌─────────┬─────────────┬───────────┬────────────┐
│ hh_null │ basket_null │ prod_null │ sales_null │
│  int64  │    int64    │   int64   │   int64    │
├─────────┼─────────────┼───────────┼────────────┤
│       0 │           0 │         0 │          0 │
└─────────┴─────────────┴───────────┴────────────┘


--- duplicate rows
┌────────────┬───────────────┐
│ total_rows │ distinct_rows │
│   int64    │     int64     │
├────────────┼───────────────┤
│    2595732 │       2595732 │
└────────────┴───────────────┘


--- is (BASKET_ID, PRODUCT_ID) unique?
┌────────────┬───────────────┐
│ total_rows │ distinct_keys │
│   int64    │     int64     │
├────────────┼───────────────┤
│    2595732 │       2595732 │
└────────────┴───────────────┘


--- DAY Range
┌────────────┬────────────┬───────────────┐
│ min("DAY") │ max("DAY") │ distinct_days │
│   int64    │   int64    │     int64     │
├────────────┼────────────┼───────────────┤
│          1 │        711 │           711 │
└─

## Duplicate Checks. For coupon and causal tables.

In [13]:
q("coupon: total rows vs unique rows",f"""
    SELECT COUNT(*) AS rows_total,
           (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM {src('coupon')})) AS unique_rows
    FROM {src('coupon')}""")

q("causal: duplicate keys, exact or conflicting?", f"""
    SELECT COUNT(*) AS duplicate_keys,
           COUNT(*) FILTER (WHERE variants = 1) AS exact_copies,
           COUNT(*) FILTER (WHERE variants > 1) AS conflicting
    FROM (
        SELECT COUNT(DISTINCT (display, mailer)) AS variants
        FROM {src('causal_data')}
        GROUP BY PRODUCT_ID, STORE_ID, WEEK_NO
        HAVING COUNT(*) > 1)""")


--- coupon: total rows vs unique rows
┌────────────┬─────────────┐
│ rows_total │ unique_rows │
│   int64    │    int64    │
├────────────┼─────────────┤
│     124548 │      119384 │
└────────────┴─────────────┘


--- causal: duplicate keys, exact or conflicting?
┌────────────────┬──────────────┬─────────────┐
│ duplicate_keys │ exact_copies │ conflicting │
│     int64      │    int64     │    int64    │
├────────────────┼──────────────┼─────────────┤
│          15245 │            0 │       15245 │
└────────────────┴──────────────┴─────────────┘



## Reference tables

In [11]:
## Products
q("product: rows vs distinct ids",
  f"SELECT COUNT(*) AS n_rows, COUNT(DISTINCT PRODUCT_ID) AS n_ids FROM {src('product')}")

empty_counts("product")

q("product: departments",
  f"SELECT DEPARTMENT, COUNT(*) AS n FROM {src('product')} GROUP BY 1 ORDER BY 2 DESC")

q("product: brand values",
  f"SELECT BRAND, COUNT(*) AS n FROM {src('product')} GROUP BY 1 ORDER BY 2 DESC")

q("sold products missing from product",
  f"""SELECT COUNT(DISTINCT t.PRODUCT_ID) AS missing_products
      FROM {src('transaction_data')} t
      LEFT JOIN {src('product')} p ON t.PRODUCT_ID = p.PRODUCT_ID
      WHERE p.PRODUCT_ID IS NULL""")


# Hh_demographic
q("hh_demographic: rows vs distinct households",
  f"SELECT COUNT(*) AS n_rows, COUNT(DISTINCT household_key) AS n_ids FROM {src('hh_demographic')}")

empty_counts("hh_demographic")

CATEGORY_COLS = ["AGE_DESC", "MARITAL_STATUS_CODE", "INCOME_DESC", "HOMEOWNER_DESC",
                 "HH_COMP_DESC", "HOUSEHOLD_SIZE_DESC", "KID_CATEGORY_DESC"]

union = " UNION ALL ".join(
    f"SELECT '{c}' AS col, CAST({c} AS VARCHAR) AS value, COUNT(*) AS n "
    f"FROM {src('hh_demographic')} GROUP BY 2"
    for c in CATEGORY_COLS
)
q("hh_demographic: value counts", f"SELECT * FROM ({union}) ORDER BY col, n DESC")

q("households in sales vs with demographics",
  f"""SELECT COUNT(DISTINCT t.household_key) AS households_in_sales,
             COUNT(DISTINCT d.household_key) AS with_demographics
      FROM {src('transaction_data')} t
      LEFT JOIN {src('hh_demographic')} d ON t.household_key = d.household_key""")


--- product: rows vs distinct ids
┌────────┬───────┐
│ n_rows │ n_ids │
│ int64  │ int64 │
├────────┼───────┤
│  92353 │ 92353 │
└────────┴───────┘


--- product: empty values per column
┌────────────┬──────────────┬────────────┬───────┬────────────────┬────────────────────┬──────────────────────┐
│ PRODUCT_ID │ MANUFACTURER │ DEPARTMENT │ BRAND │ COMMODITY_DESC │ SUB_COMMODITY_DESC │ CURR_SIZE_OF_PRODUCT │
│   int64    │    int64     │   int64    │ int64 │     int64      │       int64        │        int64         │
├────────────┼──────────────┼────────────┼───────┼────────────────┼────────────────────┼──────────────────────┤
│          0 │            0 │         15 │     0 │             15 │                 15 │                30607 │
└────────────┴──────────────┴────────────┴───────┴────────────────┴────────────────────┴──────────────────────┘


--- product: departments
┌─────────────────┬───────┐
│   DEPARTMENT    │   n   │
│     varchar     │ int64 │
├─────────────────┼───────┤
│

## Promotion tables

In [12]:
# ---------- campaign_desc and campaign_table ----------
q("campaign_desc: all rows", f"SELECT * FROM {src('campaign_desc')} ORDER BY CAMPAIGN")
q("campaign_desc: day sanity",
  f"""SELECT MIN(START_DAY) AS first_day, MAX(END_DAY) AS last_day,
             COUNT(*) FILTER (WHERE START_DAY > END_DAY) AS start_after_end
      FROM {src('campaign_desc')}""")
q("campaign_table: rows vs distinct (household, campaign)",
  f"""SELECT COUNT(*) AS n_rows,
             COUNT(DISTINCT (household_key, CAMPAIGN)) AS n_keys
      FROM {src('campaign_table')}""")
q("campaign_table: campaigns missing from campaign_desc",
  f"""SELECT COUNT(DISTINCT t.CAMPAIGN) AS missing
      FROM {src('campaign_table')} t
      LEFT JOIN {src('campaign_desc')} d ON t.CAMPAIGN = d.CAMPAIGN
      WHERE d.CAMPAIGN IS NULL""")

# ---------- coupon and coupon_redempt ----------
empty_counts("coupon")
q("coupon: rows vs distinct (coupon, product, campaign)",
  f"""SELECT COUNT(*) AS n_rows,
             COUNT(DISTINCT (COUPON_UPC, PRODUCT_ID, CAMPAIGN)) AS n_keys
      FROM {src('coupon')}""")
q("coupon_redempt: rows, distinct rows, day range",
  f"""SELECT COUNT(*) AS n_rows,
             (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM {src('coupon_redempt')})) AS distinct_rows,
             MIN(DAY) AS first_day, MAX(DAY) AS last_day
      FROM {src('coupon_redempt')}""")
q("coupon_redempt: coupons missing from coupon",
  f"""SELECT COUNT(*) AS missing
      FROM {src('coupon_redempt')} r
      LEFT JOIN (SELECT DISTINCT COUPON_UPC FROM {src('coupon')}) c
        ON r.COUPON_UPC = c.COUPON_UPC
      WHERE c.COUPON_UPC IS NULL""")

# ---------- causal_data ----------
empty_counts("causal_data")
q("causal_data: rows vs distinct (product, store, week)",
  f"""SELECT COUNT(*) AS n_rows,
             COUNT(DISTINCT (PRODUCT_ID, STORE_ID, WEEK_NO)) AS n_keys,
             MIN(WEEK_NO) AS first_week, MAX(WEEK_NO) AS last_week
      FROM {src('causal_data')}""")
q("causal_data: display values",
  f"SELECT display, COUNT(*) AS n FROM {src('causal_data')} GROUP BY 1 ORDER BY 2 DESC")
q("causal_data: mailer values",
  f"SELECT mailer, COUNT(*) AS n FROM {src('causal_data')} GROUP BY 1 ORDER BY 2 DESC")

# ---------- how weeks map to days (needed to join causal_data to sales) ----------
q("weeks: does each day belong to exactly one week?",
  f"""SELECT COUNT(*) AS days_in_more_than_one_week FROM (
        SELECT DAY FROM {src('transaction_data')}
        GROUP BY DAY HAVING COUNT(DISTINCT WEEK_NO) > 1)""")
q("weeks: do any weeks span more than 7 days?",
  f"""SELECT COUNT(*) AS long_weeks FROM (
        SELECT WEEK_NO FROM {src('transaction_data')}
        GROUP BY WEEK_NO HAVING MAX(DAY) - MIN(DAY) > 6)""")


--- campaign_desc: all rows
┌─────────────┬──────────┬───────────┬─────────┐
│ DESCRIPTION │ CAMPAIGN │ START_DAY │ END_DAY │
│   varchar   │  int64   │   int64   │  int64  │
├─────────────┼──────────┼───────────┼─────────┤
│ TypeB       │        1 │       346 │     383 │
│ TypeB       │        2 │       351 │     383 │
│ TypeC       │        3 │       356 │     412 │
│ TypeB       │        4 │       372 │     404 │
│ TypeB       │        5 │       377 │     411 │
│ TypeC       │        6 │       393 │     425 │
│ TypeB       │        7 │       398 │     432 │
│ TypeA       │        8 │       412 │     460 │
│ TypeB       │        9 │       435 │     467 │
│ TypeB       │       10 │       463 │     495 │
│ TypeB       │       11 │       477 │     523 │
│ TypeB       │       12 │       477 │     509 │
│ TypeA       │       13 │       504 │     551 │
│ TypeC       │       14 │       531 │     596 │
│ TypeC       │       15 │       547 │     708 │
│ TypeB       │       16 │       561 │  

## Silver check

In [10]:
from src.ingestion.load_bronze import get_spark
from src.transform.silver_transactions import SILVER, QUARANTINE

spark = get_spark()
spark.sparkContext.setLogLevel("ERROR")

silver = spark.read.format("delta").load(SILVER)
quarantine = spark.read.format("delta").load(QUARANTINE)

print("Silver rows:", silver.count())
print("Quarantine rows:", quarantine.count())
silver.show(3, truncate=False)
quarantine.groupBy("quarantine_reason").count().show()

Silver rows: 2581266
Quarantine rows: 14466
+-------------+-----------+---+----------+--------+--------+-------+-----------+-----------+-----------+-----------------+----------------+-------------------+------------+----------+
|household_key|basket_id  |day|product_id|quantity|store_id|week_no|sales_value|retail_disc|coupon_disc|coupon_match_disc|transaction_date|transaction_ts     |is_zero_sale|_file_date|
+-------------+-----------+---+----------+--------+--------+-------+-----------+-----------+-----------+-----------------+----------------+-------------------+------------+----------+
|1930         |28318002693|112|1123101   |1       |364     |17     |3.0        |-2.19      |0.0        |0.0              |2024-04-21      |2024-04-21 00:42:00|false       |2024-04-21|
|49           |28318333822|112|827631    |1       |356     |17     |1.67       |-0.42      |0.0        |0.0              |2024-04-21      |2024-04-21 00:33:00|false       |2024-04-21|
|49           |28318333822|112|89310

In [14]:
spark.stop()